### Summary 

- **Load Day 2 calorie information**
  - Read the Excel file `SPCD_subject_notes_mri.xlsx` (sheet: `task_notes_Day2`).
  - Extract the columns `subject number` and `Estimated Calories`.
  - Convert `Estimated Calories` to numeric.
  - Transform `subject number` into standardized subject IDs of the form `sub-XX`.

- **Load subject information**
  - Read `subject_info.csv`, which lists subjects currently included in the analysis.
  - Import each subject's `feeding` group (savory = `a`, sweet = `w`).

- **Merge datasets**
  - Merge the calorie data and subject list on the standardized `subject` column.
  - Retain only subjects present in both datasets.

- **Apply exclusions**
  - Remove subjects in `EXCLUDE_SUBJECTS` (`sub-19`, `sub-33`).
  - Drop subjects with missing calorie values.

- **Summarize estimated calories**
  - Compute descriptive statistics (count, mean, SD, quartiles) for all included subjects.
  - Compute group-wise summaries for the savory (`a`) and sweet (`w`) groups.

- **Statistical comparison**
  - Perform an independent two-sample t-test comparing calorie intake between the savory and sweet groups.
  - Output the t-statistic and p-value.


In [12]:
from pathlib import Path
import pandas as pd
from scipy.stats import ttest_ind

# --------------------------------------
# Paths
# --------------------------------------
study_path = Path('/Users/liuq13/SPCD_data/task_info')

notes_xlsx       = study_path / 'SPCD_subject_notes_mri.xlsx'
subject_list_csv = study_path / 'subject_info.csv'

EXCLUDE_SUBJECTS = {'sub-19', 'sub-33'}

# --------------------------------------
# Read calories from Excel
# --------------------------------------
df_cal = pd.read_excel(
    notes_xlsx,
    sheet_name='task_notes_Day2',
    usecols=['subject number', 'Estimated Calories']
)

df_cal['Estimated Calories'] = pd.to_numeric(df_cal['Estimated Calories'], errors='coerce')
df_cal['subject'] = df_cal['subject number'].astype(int).apply(lambda x: f"sub-{x:02d}")

# --------------------------------------
# Read subject list with feeding info
# --------------------------------------
df_subj = pd.read_csv(
    subject_list_csv,
    usecols=['subject', 'feeding']
)

# --------------------------------------
# Merge calories with subject list
# --------------------------------------
df_merge = pd.merge(df_cal, df_subj, on='subject', how='inner')

# Apply exclusions
df_merge = df_merge[~df_merge['subject'].isin(EXCLUDE_SUBJECTS)]

# Remove NaN calories
df_merge_valid = df_merge.dropna(subset=['Estimated Calories'])

print(df_merge_valid)

# --------------------------------------
# Summary
# --------------------------------------
print("\nOverall summary of Estimated Calories:")
print(df_merge_valid['Estimated Calories'].describe())

print("\nGroup means:")
print(df_merge_valid.groupby('feeding')['Estimated Calories'].mean())

print("\nGroup summaries:")
print(df_merge_valid.groupby('feeding')['Estimated Calories'].describe())

# --------------------------------------
# T-test: a vs w
# --------------------------------------
a_vals = df_merge_valid[df_merge_valid['feeding'] == 'a']['Estimated Calories']
w_vals = df_merge_valid[df_merge_valid['feeding'] == 'w']['Estimated Calories']

t_stat, p_val = ttest_ind(a_vals, w_vals, nan_policy='omit')

print("\nT-test comparing Estimated Calories: a vs w")
print(f"t = {t_stat:.4f},  p = {p_val:.4f}")


    subject number  Estimated Calories subject feeding
0                1               650.0  sub-01       w
1                2               910.0  sub-02       w
2                3               900.0  sub-03       a
3                4              1200.0  sub-04       a
4                7              1237.5  sub-07       a
5                9               875.0  sub-09       w
6               10               780.0  sub-10       w
7               11              1088.2  sub-11       a
8               13               585.0  sub-13       w
9               14               750.0  sub-14       w
10              15               520.0  sub-15       a
11              16               600.0  sub-16       w
12              17               660.0  sub-17       a
14              20               610.0  sub-20       w
15              22               447.8  sub-22       w
16              23               696.0  sub-23       a
17              25              1000.0  sub-25       w
18        